In [5]:
import pandas as pd
import geopandas as gpd

# ── Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv('/Users/jackzipper/QSS20/final_project/final_project_data/iati-activity-locations-in-democratic-republic-of-the-congo.csv')
provinces = gpd.read_file('/Users/jackzipper/QSS20/final_project/final_project_data/cod_admin_boundaries.shp/cod_admin1.shp')
admin2    = gpd.read_file('/Users/jackzipper/QSS20/final_project/final_project_data/cod_admin_boundaries.shp/cod_admin2.shp')

# ── Project everything to meters-based CRS for accurate distance calculations
provinces_proj = provinces.to_crs('EPSG:32635')
admin2_proj    = admin2.to_crs('EPSG:32635')

# ── Build projected GeoDataFrame of aid points ────────────────────────────
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['location_longitude'], df['location_latitude']),
    crs='EPSG:4326'
).to_crs('EPSG:32635')

# ══════════════════════════════════════════════════════════════════════════
# PART 1 — Admin-1 Province
# ══════════════════════════════════════════════════════════════════════════

# Exact point-in-polygon join
joined = gpd.sjoin(gdf, provinces_proj[['adm1_name', 'geometry']], how='left', predicate='within')
joined = joined.drop(columns=['index_right'])
joined['province_name'] = joined['adm1_name']

# For remaining NAs use nearest province within ~55 km
na_mask = joined['province_name'].isna()
gdf_na = gpd.GeoDataFrame(
    joined[na_mask].drop(columns=['adm1_name']),
    geometry=gpd.points_from_xy(
        joined.loc[na_mask, 'location_longitude'],
        joined.loc[na_mask, 'location_latitude']
    ),
    crs='EPSG:4326'
).to_crs('EPSG:32635')

nearest_adm1 = gpd.sjoin_nearest(gdf_na, provinces_proj[['adm1_name', 'geometry']], how='left', max_distance=55000)
joined.loc[nearest_adm1.index, 'province_name'] = nearest_adm1['adm1_name'].values

# Drop rows still NA (outside DRC) and clean up
df_final = joined.drop(columns=['adm1_name', 'geometry'])
df_final = df_final.dropna(subset=['province_name'])

# ══════════════════════════════════════════════════════════════════════════
# PART 2 — Admin-2 Territory/Town
# ══════════════════════════════════════════════════════════════════════════

# Auto-detect the Admin-2 name column
CANDIDATE_NAME_COLS = ['admin2Name', 'ADM2_EN', 'ADM2_FR', 'NAME_2',
                       'name', 'NAME', 'adm2_name', 'admin2name']
adm2_name_col = next((c for c in CANDIDATE_NAME_COLS if c in admin2.columns), None)
if adm2_name_col is None:
    raise ValueError(
        f"Could not auto-detect Admin-2 name column.\n"
        f"Available columns: {admin2.columns.tolist()}\n"
        f"Set adm2_name_col manually."
    )
print(f"Admin-2 name column detected: '{adm2_name_col}'")

# Rebuild projected GeoDataFrame from df_final (province-matched rows only)
gdf2 = gpd.GeoDataFrame(
    df_final,
    geometry=gpd.points_from_xy(df_final['location_longitude'], df_final['location_latitude']),
    crs='EPSG:4326'
).to_crs('EPSG:32635')

# Exact point-in-polygon join against Admin-2
joined2 = gpd.sjoin(gdf2, admin2_proj[[adm2_name_col, 'geometry']], how='left', predicate='within')
joined2 = joined2.drop(columns=['index_right'])
joined2['admin2_name'] = joined2[adm2_name_col]

# For remaining NAs use nearest Admin-2 polygon within ~55 km
na_mask2 = joined2['admin2_name'].isna()
gdf2_na = gpd.GeoDataFrame(
    joined2[na_mask2].drop(columns=[adm2_name_col]),
    geometry=gpd.points_from_xy(
        joined2.loc[na_mask2, 'location_longitude'],
        joined2.loc[na_mask2, 'location_latitude']
    ),
    crs='EPSG:4326'
).to_crs('EPSG:32635')

nearest_adm2 = gpd.sjoin_nearest(gdf2_na, admin2_proj[[adm2_name_col, 'geometry']], how='left', max_distance=55000)
joined2.loc[nearest_adm2.index, 'admin2_name'] = nearest_adm2[adm2_name_col].values

# Clean up geometry columns
df_final = joined2.drop(columns=[adm2_name_col, 'geometry'], errors='ignore')
df_final = pd.DataFrame(df_final)

# ══════════════════════════════════════════════════════════════════════════
# PART 3 — Deduplicate
# ══════════════════════════════════════════════════════════════════════════

print(f"Before dedup: {len(df_final)} rows")

# Step 1: drop exact duplicates
df_final = df_final.drop_duplicates()
print(f"After dropping exact dupes: {len(df_final)} rows")

# Step 2: drop remaining dupes on key columns. Some columns have different longitude and latitudes,
# meaning that they are the same aid project in different locations in the same province. This would
# not get picked up on by the drop_duplicates() method.
key_cols = ['aid', 'province_name', 'day_start', 'day_end', 'description', 'spend']
df_final = df_final.drop_duplicates(subset=key_cols)
print(f"After dropping key col dupes: {len(df_final)} rows")

print(f"Unique aid projects: {df_final['aid'].nunique()}")
print(f"\nNAs in province_name: {df_final['province_name'].isna().sum()}")
print(f"NAs in admin2_name  : {df_final['admin2_name'].isna().sum()}")
print(f"Unique provinces    : {df_final['province_name'].nunique()}")
print(f"Unique admin2 towns : {df_final['admin2_name'].nunique()}")
print()
print(df_final['province_name'].value_counts())
print()
print(df_final['admin2_name'].value_counts().head(20))

# ── Save ───────────────────────────────────────────────────────────────────
df_final.to_csv('/Users/jackzipper/QSS20/final_project/final_project_data/iati-drc-cleaned.csv', index=False)
print("\nSaved to iati-drc-cleaned.csv")

Admin-2 name column detected: 'adm2_name'
Before dedup: 28324 rows
After dropping exact dupes: 6332 rows
After dropping key col dupes: 4917 rows
Unique aid projects: 2836

NAs in province_name: 0
NAs in admin2_name  : 0
Unique provinces    : 26
Unique admin2 towns : 148

province_name
Kinshasa          1338
Nord-Kivu          445
Kasaï              434
Sud-Kivu           394
Sankuru            249
Ituri              229
Tanganyika         198
Haut-Katanga       174
Tshopo             173
Kasaï-Oriental     145
Maniema            134
Kasaï-Central      133
Kongo-Central      110
Lomami              85
Equateur            70
Lualaba             70
Haut-Lomami         70
Bas-Uele            70
Kwilu               69
Sud-Ubangi          62
Kwango              52
Nord-Ubangi         49
Haut-Uele           48
Maï-Ndombe          39
Tshuapa             39
Mongala             38
Name: count, dtype: int64

admin2_name
Kinshasa        1338
Mweka            278
Goma             247
Bukavu        